#### <font color='#0000DD'><b> Welcome to the prototype demonstration for the paper *KRAFT -- A Knowledge-Graph-Based Resource Allocation Framework*
<font color='#0000DD'><b> This Jupyter Notebook walks you through the features of the prototype and framework. Marked in blue is contextual information and instructions.

# Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from rdflib import Graph, Literal, RDF, URIRef, Namespace
from urllib.parse import quote, unquote
import pandas as pd
from datetime import timedelta

In [3]:
import sys  
import os
root = os.path.abspath('')
sys.path.insert(1, root + '/Business Process Optimization Competition 2023')
from simulator import Simulator
from problems import MinedProblem
sys.path.insert(1, root + '/src')
from KGPlanner import KGPlanner, Mode
from ProcessKnowledgeGraph import ProcessKnowledgeGraph
from SHACLAllocator import SHACLAllocator 
from SimulatorForDemonstration import SimulatorForDemonstration
from utils import *

# Mining Resource Expertise
<font color='#0000DD'><b>The following demonstrates how process mining techniques can be utilized to extract knowledge about resources, in this case expertise for specific loan goals derived from the RBI 1.2 "Case Types" from Pika et al., "Mining Resource Profiles from Event Logs" (2017).
First, the BPIC log is read from file and the respective metrics (cases completed per resource in total and further divided per loan goal) are calculated.
Then, expertise for specific loan goals is derived via a threshold from the RBI.

In [4]:
import pandas as pd
import numpy as np

In [5]:
log = pd.read_csv('./BPI Challenge 2017 - clean.csv')
min_date = log['Start Timestamp'].min()
max_date = log['Complete Timestamp'].max()
total_cases_per_resource = log.groupby(['Resource']).agg({'Case ID' : pd.Series.nunique})['Case ID']
cases_per_resource_per_loan_goal = log.groupby(['LoanGoal', 'Resource']).agg({'Case ID' : pd.Series.nunique}).unstack()['Case ID']
cases_per_resource_per_loan_goal_fraction = cases_per_resource_per_loan_goal.div(total_cases_per_resource.transpose()).fillna(0) # The RBI 1.2
experts = cases_per_resource_per_loan_goal_fraction > 0.35

<font color='#0000DD'><b> To translate this knowledge to graph, nodes for the resources and loan goals as well as appropriate `expertOn` edges are created:

In [6]:
helper_graph = ProcessKnowledgeGraph()
for loan_goal in experts.index:
    helper_graph.add_entity('LoanGoal', loan_goal)
    for resource in experts.columns:
        if experts.loc[loan_goal][resource]:
            helper_graph.add_entity('resource', resource)
            helper_graph.add_from_strings('resource:'+resource, 'expertOn', 'LoanGoal:'+loan_goal)

<font color='#0000DD'><b> We further determine seniority of the resources based on the number of case completions

In [7]:
seniorities = total_cases_per_resource.apply(lambda cpr : 'High' if cpr > 1500 else ('Medium' if cpr > 300 else 'Low'))
for resource, seniority in seniorities.items():
    helper_graph.add_entity('resource', resource) # Redundant for resources that are not experts on anything; will not produce duplicates
    helper_graph.add_entity('Seniority', seniority)
    helper_graph.add_from_strings('resource:'+resource, 'Seniority', 'Seniority:'+seniority)

<font color='#0000DD'><b>  You can have a look at the generated graph as follows. Note that you can put this viewer into fullscreen mode, and you can explore a node's neighbourhood by selecting a node and opening the sidebar.

In [8]:
draw_graph(helper_graph)

GraphWidget(layout=Layout(height='800px', width='100%'))

<font color='#0000DD'><b> Now write the knowledge to file for later use:

In [9]:
helper_graph.serialize(destination='./src/extension_knowledge/expertise_instances.ttl')

<Graph identifier=Nf956ac6dc96a446087429e9eabaca510 (<class 'ProcessKnowledgeGraph.ProcessKnowledgeGraph'>)>

# Example Simulation
<font color='#0000DD'><b>Execute the following three steps to run the example simulation. Don't rerun single steps.

### Planner & Allocator Setup
<font color='#0000DD'><b>The planner is the interface to the simulation frame, implementing a function to assign a set of open tasks.
The allocation actually manages the reasoning based on the encoded ontology and rules. In this example setup, we load additional knowledge as described in the paper's example use case.
Note that we reuse the knowledge earlier derived from Process Mining by loading the respective file.

In [128]:
planner = KGPlanner()
planner.allocator.load_extension(ontology_ext='./src/extension_knowledge/extension_ontology.ttl')
planner.allocator.load_extension(rules_ext='./src/extension_knowledge/sepOfConcerns_rules.ttl')
planner.allocator.load_extension(rules_ext='./src/extension_knowledge/riskAndSeniority_rules.ttl', instance_ext='./src/extension_knowledge/riskAndSeniority_instances.ttl')
planner.allocator.load_extension(rules_ext='./src/extension_knowledge/expertise_rules.ttl', instance_ext='./src/extension_knowledge/expertise_instances.ttl')

<font color='#0000DD'><b> At this point, and at any point when the simulation is paused, it is possible to manually add further knowledge. 
For example, let's manually define some expertises that have not been discovered by the process mining techniques: 

In [129]:
planner.graph.add_from_strings('resource:User_24', 'expertOn', 'LoanGoal:Remaining debt home')
planner.graph.add_from_strings('resource:User_99', 'expertOn', 'LoanGoal:Boat')
planner.graph.add_from_strings('resource:User_123', 'expertOn', 'LoanGoal:Extra spending limit')
planner.graph.add_from_strings('resource:User_56', 'expertOn', 'LoanGoal:Home improvement')

### Simulation Frame Setup
<font color='#0000DD'><b>The simulation frame and simulated process are taken from the BPOC 2023. We adapt the simulation numbers as to allow for more interesting allocation situations.

In [130]:
simulator = SimulatorForDemonstration(planner=planner, instance_file='./Business Process Optimization Competition 2023/BPI Challenge 2017 - instance 2.pickle')
problem = simulator.problem
# Increase Arrival Rate so also more resources are provided
problem.interarrival_time._scale *= 0.5
# Increase likelihood for assessing fraud, so separation of concerns actually triggers
problem.next_task_distribution['W_Validate application'] = list(map(lambda x : (x[0] * 0.8 + (0.2 if (x[1] == 'W_Assess potential fraud') else 0), x[1]), problem.next_task_distribution['W_Validate application']))

### Run the Simulation

<font color='#0000DD'><b>First, let's run just run the initial first task assignment of the simulation. The console logs show current allocation context (case, task, and activity as well as available and pool-allowed resources), the decision made by the system and the respective reasoning. 

In [131]:
simulator.run(0)


0 case-0 task-0: W_Complete application 
Resources Available: {'User_70', 'User_85', 'User_146', 'User_47', 'User_140'} 
Resources in Pool: {'User_47', 'User_70', 'User_85'}
Assigning: User_70 to task-0 considering the following: 
 	Seniority 'Medium' is sufficient for risk class 'Low' of loan goal 'Home improvement'
	Resource has relevant role(s) for activity 'W_Complete application'

User_70 is now an expert on application type New credit


(0.0,
 'COMPLETED: you completed 0 hours of simulated customer cases. 1 cases started. 0 cases run to completion.')

<font color='#0000DD'><b>We can now investigate the current state of the allocation knowledge graph after one allocation decision. 
To ensure that we only see the resources currently present, we filter accordingly.  
This is a nice illustration of the relevant entities for one single decision (although some background knowledge entities are visible as well as they are already loaded).
Note that entities are color-coded by their type. 

In [132]:
draw_graph(planner.graph.subgraph_available_resources())

GraphWidget(layout=Layout(height='800px', width='100%'))

<font color='#0000DD'><b> Let's resume the simulation to run 1 more hour of simulation time. This will take some (real) time. However, the simulation can be paused at any moment using Jupyter's interrupt function (the interrupt button or pressing `i` twice while having the respective cell selected.) 

In [133]:
simulator.run(1)


0.09008078297095312 case-0 task-2: W_Complete application 
Resources Available: {'User_70', 'User_85', 'User_146', 'User_47', 'User_140'} 
Resources in Pool: {'User_47', 'User_70', 'User_85'}
Assigning: User_70 to task-2 considering the following: 
 	Seniority 'Medium' is sufficient for risk class 'Low' of loan goal 'Home improvement'
	Resource is an expert on 'ApplicationType' 'New credit'
	Resource has relevant role(s) for activity 'W_Complete application'


0.17995947185881933 case-0 task-3: W_Call after offers 
Resources Available: {'User_70', 'User_85', 'User_146', 'User_47', 'User_140'} 
Resources in Pool: {'User_47', 'User_70', 'User_85'}
Assigning: User_70 to task-3 considering the following: 
 	Seniority 'Medium' is sufficient for risk class 'Low' of loan goal 'Home improvement'
	Resource is an expert on 'ApplicationType' 'New credit'
	Resource has relevant role(s) for activity 'W_Call after offers'


0.29207814462803483 case-0 task-4: W_Call after offers 
Resources Available

(0.2913405759499653,
 'COMPLETED: you completed 1.0900807829709531 hours of simulated customer cases. 14 cases started. 3 cases run to completion.')

<font color='#0000DD'><b> While paused, we can add more knowledge, and also change the system mode to human-in-the-loop (HITL).
In HITL, the system prompts users for every allocation decision. We can then continue the simulation a little more in this mode.

In [134]:
planner.graph.add_entity('ApplicationType', 'Limit raise')
planner.graph.add_from_strings('resource:'+resource, 'expertOn', 'ApplicationType:Limit raise')
planner.mode = Mode.HUMAN_IN_THE_LOOP
simulator.run(0.1)


1.1337856094570307 case-10 task-36: W_Call after offers 
Resources Available: {'User_97', 'User_140'} 
Resources in Pool: {'User_97'}
The following resources are available:
0: User_97, score: 2, considering: 
 	Seniority 'Low' is sufficient for risk class 'Low' of loan goal 'Home improvement'
	Resource has relevant role(s) for activity 'W_Call after offers'

1: User_140, score: -inf, considering: 
 	Seniority 'Low' is sufficient for risk class 'Low' of loan goal 'Home improvement'
	The resource 'User_140' is not allowed to perform the activity 'W_Call after offers'

Type the index of the resource to select. Type -1 to not assign any resource.


 0


Assigning: User_97 to task-36 considering the following: 
 	Seniority 'Low' is sufficient for risk class 'Low' of loan goal 'Home improvement'
	Resource has relevant role(s) for activity 'W_Call after offers'


1.1675080425332238 case-4 task-37: W_Call incomplete files 
Resources Available: {'User_85', 'User_140'} 
Resources in Pool: {'User_85'}
The following resources are available:
0: User_85, score: -1, considering: 
 	Seniority 'Medium' not sufficient for risk class 'High' of loan goal 'Car'
	Resource has relevant role(s) for activity 'W_Call incomplete files'

1: User_140, score: -inf, considering: 
 	Seniority 'Low' not sufficient for risk class 'High' of loan goal 'Car'
	The resource 'User_140' is not allowed to perform the activity 'W_Call incomplete files'

Type the index of the resource to select. Type -1 to not assign any resource.


Assigning: User_85 to task-37 considering the following: 
 	Seniority 'Medium' not sufficient for risk class 'High' of loan goal 'Car'
	Resource has relevant role(s) for activity 'W_Call incomplete files'


1.1716983190590837 case-14 task-35: W_Complete application 
Resources Available: {'User_96', 'User_140'} 
Resources in Pool: {'User_96'}
The following resources are available:
0: User_96, score: 2, considering: 
 	Seniority 'Medium' is sufficient for risk class 'Low' of loan goal 'Other, see explanation'
	Resource has relevant role(s) for activity 'W_Complete application'

1: User_140, score: -inf, considering: 
 	Seniority 'Low' is sufficient for risk class 'Low' of loan goal 'Other, see explanation'
	The resource 'User_140' is not allowed to perform the activity 'W_Complete application'

Type the index of the resource to select. Type -1 to not assign any resource.


Assigning: User_96 to task-35 considering the following: 
 	Seniority 'Medium' is sufficient for risk class 'Low' of loan goal 'Other, see explanation'
	Resource has relevant role(s) for activity 'W_Complete application'


1.1795967799120493 case-5 task-30: W_Validate application 
Resources Available: {'User_146', 'User_140'} 
Resources in Pool: set()
The following resources are available:
0: User_140, score: -inf, considering: 
 	Seniority 'Low' is sufficient for risk class 'Low' of loan goal 'Home improvement'
	The resource 'User_140' is not allowed to perform the activity 'W_Validate application'

1: User_146, score: -inf, considering: 
 	Seniority 'Low' is sufficient for risk class 'Low' of loan goal 'Home improvement'
	The resource 'User_146' is not allowed to perform the activity 'W_Validate application'

Type the index of the resource to select. Type -1 to not assign any resource.


...

Invalid resource, cannot allocate. Skip.
No assignment made

1.1905044636735842 case-11 task-41: W_Complete application 
Resources Available: {'User_146', 'User_140'} 
Resources in Pool: set()
The following resources are available:
0: User_146, score: -inf, considering: 
 	Seniority 'Low' not sufficient for risk class 'High' of loan goal 'Existing loan takeover'
	The resource 'User_146' is not allowed to perform the activity 'W_Complete application'

1: User_140, score: -inf, considering: 
 	Seniority 'Low' not sufficient for risk class 'High' of loan goal 'Existing loan takeover'
	The resource 'User_140' is not allowed to perform the activity 'W_Complete application'

Type the index of the resource to select. Type -1 to not assign any resource.


Invalid resource, cannot allocate. Skip.
No assignment made

1.1905044636735842 case-13 task-32: W_Complete application 
Resources Available: {'User_146', 'User_140'} 
Resources in Pool: set()
The following resources are available:
0: User_140, score: -inf, considering: 
 	Seniority 'Low' not sufficient for risk class 'High' of loan goal 'Existing loan takeover'
	The resource 'User_140' is not allowed to perform the activity 'W_Complete application'

1: User_146, score: -inf, considering: 
 	Seniority 'Low' not sufficient for risk class 'High' of loan goal 'Existing loan takeover'
	The resource 'User_146' is not allowed to perform the activity 'W_Complete application'

Type the index of the resource to select. Type -1 to not assign any resource.


Invalid resource, cannot allocate. Skip.
No assignment made

1.1905044636735842 case-2 task-40: W_Validate application 
Resources Available: {'User_146', 'User_140'} 
Resources in Pool: set()
The following resources are available:
0: User_140, score: -inf, considering: 
 	Seniority 'Low' not sufficient for risk class 'High' of loan goal 'Car'
	The resource 'User_140' is not allowed to perform the activity 'W_Validate application'

1: User_146, score: -inf, considering: 
 	Seniority 'Low' not sufficient for risk class 'High' of loan goal 'Car'
	The resource 'User_146' is not allowed to perform the activity 'W_Validate application'

Type the index of the resource to select. Type -1 to not assign any resource.


Invalid resource, cannot allocate. Skip.
No assignment made


(0.31708883105423674,
 'COMPLETED: you completed 1.1963965367807996 hours of simulated customer cases. 27 cases started. 14 cases run to completion.')

<font color='#0000DD'><b> Afterward, we can reset the mode and continue the simulation for some more hours.

In [135]:
planner.mode = Mode.AUTOMATED
simulator.run(2)


1.2011344077714512 case-10 task-42: W_Call after offers 
Resources Available: {'User_146', 'User_97', 'User_140'} 
Resources in Pool: {'User_97'}
Assigning: User_97 to task-42 considering the following: 
 	Seniority 'Low' is sufficient for risk class 'Low' of loan goal 'Home improvement'
	Resource has relevant role(s) for activity 'W_Call after offers'


1.2011344077714512 case-7 task-39: W_Call after offers 
Resources Available: {'User_146', 'User_140'} 
Resources in Pool: set()
No assignment made

1.2011344077714512 case-6 task-33: W_Complete application 
Resources Available: {'User_146', 'User_140'} 
Resources in Pool: set()
No assignment made

1.2011344077714512 case-8 task-25: W_Complete application 
Resources Available: {'User_146', 'User_140'} 
Resources in Pool: set()
No assignment made

1.2011344077714512 case-9 task-34: W_Call after offers 
Resources Available: {'User_146', 'User_140'} 
Resources in Pool: set()
No assignment made

1.2011344077714512 case-11 task-41: W_Comple

(0.8182073688212546,
 'COMPLETED: you completed 3.201134407771451 hours of simulated customer cases. 42 cases started. 32 cases run to completion.')

<font color='#0000DD'><b> Finally, we can have a look at the whole final graph.

In [136]:
draw_graph(planner.graph)

GraphWidget(layout=Layout(height='800px', width='100%'))

# Dev

In [20]:
raise Exception('Don\'t just jump into dev')


KeyboardInterrupt



In [82]:
from pyshacl import validate

In [83]:
print(inspect.getsource(validate))

def validate(
    data_graph: Union[GraphLike, BufferedIOBase, TextIOBase, str, bytes],
    *args,
    shacl_graph: Optional[Union[GraphLike, BufferedIOBase, TextIOBase, str, bytes]] = None,
    ont_graph: Optional[Union[GraphLike, BufferedIOBase, TextIOBase, str, bytes]] = None,
    advanced: Optional[bool] = False,
    inference: Optional[str] = None,
    inplace: Optional[bool] = False,
    abort_on_first: Optional[bool] = False,
    allow_infos: Optional[bool] = False,
    allow_warnings: Optional[bool] = False,
    max_validation_depth: Optional[int] = None,
    sparql_mode: Optional[bool] = False,
    focus_nodes: Optional[List[Union[str, URIRef]]] = None,
    use_shapes: Optional[List[Union[str, URIRef]]] = None,
    **kwargs,
):
    """
    :param data_graph: rdflib.Graph or file path or web url of the data to validate
    :type data_graph: rdflib.Graph | str | bytes
    :param args:
    :type args: list
    :param shacl_graph: rdflib.Graph or file path or web url of the SHACL 

In [ ]:
from rdflib import URIRef

In [85]:
xplanner = KGPlanner()
xplanner.graph = (planner.graph+Graph())
xplanner.allocator.graph_to_check = xplanner.graph
#xplanner.allocator.load_extension(ontology_ext='./src/extension_knowledge/extension_ontology.ttl')
#xplanner.allocator.load_extension(rules_ext='./src/extension_knowledge/sepOfConcerns_rules.ttl')
#xplanner.allocator.load_extension(rules_ext='./src/extension_knowledge/riskAndSeniority_rules.ttl', instance_ext='./src/extension_knowledge/riskAndSeniority_instances.ttl')
#xplanner.allocator.load_extension(rules_ext='./src/extension_knowledge/expertise_rules.ttl', instance_ext='./src/extension_knowledge/expertise_instances.ttl')

In [72]:
xplanner.graph.uri('resource:User_5')

rdflib.term.URIRef('http://example.org/instances/resources/User_5')

In [79]:
for rule in xplanner.allocator.shacl_graph.subjects(URIRef('http://www.w3.org/ns/shacl#targetClass')):
    print(rule)
#    xplanner.allocator.shacl_graph.add((rule, URIRef('http://www.w3.org/ns/shacl#targetNode'), xplanner.graph.uri('resource:User_5')))

http://foobar.org/ExecutionPermissionConstraint
http://foobar.org/ExecutionPermissionReward


In [81]:
len(xplanner.graph)

578

In [86]:
%%time
xplanner.determine_resource(list(xplanner.graph.unassigned_tasks())[0])

CPU times: user 204 ms, sys: 47 μs, total: 204 ms
Wall time: 204 ms


(-inf, None, 'No fitting resource found')

In [61]:
import inspect
print(inspect.getsource(xplanner.allocator.test_assignment))

    def test_assignment(self, task_node, resource_node):
        hypothetical = (
            (task_node,
             self.graph_to_check.attribute_relation(Keys.RESOURCE) + ('__hypothetical' if self.use_hypothetical else ''), #TODO magic string
             resource_node)
        )
    
        assert hypothetical not in self.graph_to_check
        try:
            self.graph_to_check.add(hypothetical)
            
            r = validate(self.graph_to_check,
                  shacl_graph=self.shacl_graph,
                  ont_graph=self.ontology,
                  inference=None, # TODO 'both',
                  abort_on_first=False,
                  allow_infos=True,
                  allow_warnings=True,
                  meta_shacl=True,
                  advanced=True,
                  js=False,
                  debug=False)
        finally:
            self.graph_to_check.remove(hypothetical)
        
        # display(r)
        # print(results_text)
        return r



In [66]:
draw_graph(planner.graph)

GraphWidget(layout=Layout(height='800px', width='100%'))

In [26]:
%load_ext line_profiler

In [45]:
planner.graph.update_availability(lambda resource_node: uri_to_id(resource_node) in {'User_37', 'User_34', 'User_144', 'User_66', 'User_12'})

In [48]:
planner = simulator.planner
allocator = planner.allocator

In [50]:
%lprun -f allocator.test_assignment planner.determine_resource(list(planner.graph.unassigned_tasks())[0])

Timer unit: 1e-09 s

Total time: 107.901 s
File: /mnt/c/unsynchronized/projects/.Graph Workbench/Workbench/knowledge-graph-resource-allocation/src/SHACLAllocator.py
Function: test_assignment at line 97

Line #      Hits         Time  Per Hit   % Time  Line Contents
    97                                               def test_assignment(self, task_node, resource_node):
    98         5        527.0    105.4      0.0          hypothetical = (
    99        10       3128.0    312.8      0.0              (task_node,
   100         5      66329.0  13265.8      0.0               self.graph_to_check.attribute_relation(Keys.RESOURCE) + ('__hypothetical' if self.use_hypothetical else ''), #TODO magic string
   101         5        580.0    116.0      0.0               resource_node)
   102                                                   )
   103                                               
   104         5      27201.0   5440.2      0.0          assert hypothetical not in self.graph_to_che

In [ ]:
float('inf')

In [ ]:
x = simulator.problem.data_types['ApplicationType']
sorted(zip(x._weights, x._values))

In [ ]:
x = simulator.problem.data_types['LoanGoal']
sorted(zip(x._weights, x._values))

In [ ]:
print(planner.graph.serialize(format='n3'))

In [ ]:
from ProcessKnowledgeGraph import ProcessKnowledgeGraph

In [ ]:
test_graph_data = '''
@prefix ApplicationType: <http://example.org/instances/ApplicationTypes/> .
@prefix LoanGoal: <http://example.org/instances/LoanGoals/> .
@prefix activity: <http://example.org/instances/activitys/> .
@prefix case: <http://example.org/instances/cases/> .
@prefix task: <http://example.org/instances/tasks/> .
@prefix relation: <http://example.org/relations/> .
@prefix resource: <http://example.org/instances/resources/> .
@prefix four_eyes: <http://example.org/instances/four_eyes/> .
@prefix type: <http://example.org/types/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .


task:task-1 a type:task ;
    relation:directlyPrecedes task:task-2 ;
    relation:instanceOf activity:W_Call%20after%20offers ;
    relation:performedBy resource:User_1 ;
    relation:partOf case:case-0 .
    
task:task-2 a type:task ;
    relation:directlyPrecedes task:task-3 ;
    relation:instanceOf activity:W_Validate%20application ;
    relation:performedBy resource:User_2 ;
    relation:partOf case:case-0 .
    
task:task-3 a type:task ;
    relation:instanceOf activity:W_Assess%20potential%20fraud ;
    relation:partOf case:case-0 .
    

activity:W_Handle%20leads a type:activity ;
    relation:canBeExecutedBy resource:User_2,
        resource:User_1,
        resource:User_3 .

activity:W_Complete%20application a type:activity ;
    relation:canBeExecutedBy resource:User_2,
        resource:User_1,
        resource:User_3 .


activity:W_Call%20after%20offers a type:activity ;
    relation:canBeExecutedBy resource:User_2,
        resource:User_1,
        resource:User_3 .

activity:W_Call%20incomplete%20files a type:activity ;
    relation:canBeExecutedBy resource:User_2,
        resource:User_1,
        resource:User_3 .

activity:W_Validate%20application a type:activity ;
    relation:canBeExecutedBy resource:User_2,
        resource:User_1,
        resource:User_3 .

activity:W_Assess%20potential%20fraud a type:activity ;
    relation:canBeExecutedBy resource:User_2,
        resource:User_1,
        resource:User_3 .

case:case-0 a type:case ;
    relation:ApplicationType ApplicationType:New%20credit ;
    relation:LoanGoal LoanGoal:Existing%20loan%20takeover .



ApplicationType:New%20credit a type:ApplicationType .

LoanGoal:Existing%20loan%20takeover a type:LoanGoal .

resource:User_3 a type:resource ;
    relation:available true.

resource:User_2 a type:resource ;
    relation:available true.

resource:User_1 a type:resource ;
    relation:available true.
    
four_eyes:rule1 a type:four_eyes ;
    relation:contains activity:W_Validate%20application, activity:W_Assess%20potential%20fraud.


'''

test_graph = ProcessKnowledgeGraph()
test_graph.parse(data=test_graph_data, format='turtle')
# draw_graph(test_graph)

In [ ]:
_uri = test_graph.uri
# test_graph.add(test_graph.entity_triple('RiskClass', 'Low'))
# test_graph.add(test_graph.entity_triple('RiskClass', 'Medium'))
# test_graph.add(test_graph.entity_triple('RiskClass', 'High'))
# test_graph.add(test_graph.entity_triple('Seniority', 'Low'))
# test_graph.add((_uri('RiskClass:Low'), test_graph.attribute_relation('minSeniority'), _uri('Seniority:Low')))
test_graph.add((_uri('resource:User_1'), test_graph.attribute_relation('Seniority'), _uri('Seniority:Low')))
# test_graph.add(test_graph.entity_triple('Seniority', 'Medium'))
# test_graph.add((_uri('RiskClass:Medium'), test_graph.attribute_relation('minSeniority'), _uri('Seniority:Medium')))
test_graph.add((_uri('resource:User_2'), test_graph.attribute_relation('Seniority'), _uri('Seniority:Medium')))
# test_graph.add(test_graph.entity_triple('Seniority', 'High'))
# test_graph.add((_uri('RiskClass:High'), test_graph.attribute_relation('minSeniority'), _uri('Seniority:High')))
test_graph.add((_uri('resource:User_3'), test_graph.attribute_relation('Seniority'), _uri('Seniority:High')))

# test_graph.parse('./src/extension_instances.ttl', format='n3')
# test_graph.add((_uri('Seniority:High'), test_graph.attribute_relation('greaterEq'), _uri('Seniority:High')))
# test_graph.add((_uri('Seniority:Medium'), test_graph.attribute_relation('greaterEq'), _uri('Seniority:Medium')))
# test_graph.add((_uri('Seniority:Low'), test_graph.attribute_relation('greaterEq'), _uri('Seniority:Low')))
# test_graph.add((_uri('Seniority:High'), test_graph.attribute_relation('greaterEq'), _uri('Seniority:Low')))
# test_graph.add((_uri('Seniority:High'), test_graph.attribute_relation('greaterEq'), _uri('Seniority:Medium')))
# test_graph.add((_uri('Seniority:Medium'), test_graph.attribute_relation('greaterEq'), _uri('Seniority:Low')))

# test_graph.add((_uri('LoanGoal:Existing loan takeover'), test_graph.attribute_relation('RiskClass'), _uri('RiskClass:Medium')))
draw_graph(test_graph)

In [ ]:

task_to_be_allocated = test_graph.uri('task:task-3')

validator = SHACLAllocator(test_graph)
validator.shacl_graph = Graph() # reset
# validator.load_extension(rules_ext='./src/extension_riskReqSeniority.ttl')

validator.use_hypothetical = True
validator.load_extension(rules_ext='./src/extension_riskReqSeniority.ttl')
validator.load_extension(instance_ext='./src/extension_instances.ttl')

for sen_index, seniority in enumerate(['Low', 'Medium', 'High']):
    test_graph.set((_uri('LoanGoal:Existing loan takeover'), test_graph.attribute_relation('RiskClass'), _uri('RiskClass:'+seniority)))
    for res_index, resource in enumerate(['User_1', 'User_2', 'User_3']):
        print(test_graph.uri('resource:'+resource))
        print(validator.test_assignment(task_to_be_allocated, test_graph.uri('resource:'+resource))[2])

In [ ]:
# print(test_graph.serialize(format='n3'))

In [ ]:
import random
x = '\n\n'.join(list(map(lambda goal : f'LoanGoal:{quote(goal)} a type:LoanGoal ;\n    relation:RiskClass RiskClass:{random.sample(['Low', 'Medium', 'High'], 1)[0]} .', simulator.problem.data_types['LoanGoal']._values)))
print(x)



In [ ]:
test_graph = ProcessKnowledgeGraph()
test_graph.parse(data=test_graph_data, format='turtle')
validator = SHACLAllocator(test_graph)
validator.shacl_graph = Graph() # reset

validator.load_extension(rules_ext='./src/extension_riskReqSeniority.ttl')
validator.load_extension(instance_ext='./src/extension_instances.ttl')

test_graph.add((test_graph.uri('resource:User_1'), test_graph.attribute_relation('Seniority'), test_graph.uri('Seniority:Low')))
test_graph.add((test_graph.uri('resource:User_2'), test_graph.attribute_relation('Seniority'), test_graph.uri('Seniority:Medium')))
test_graph.add((test_graph.uri('resource:User_3'), test_graph.attribute_relation('Seniority'), test_graph.uri('Seniority:High')))


task_to_be_allocated = test_graph.uri('task:task-3')
for sen_index, seniority in enumerate(['Low', 'Medium', 'High']):
    test_graph.set((test_graph.uri('LoanGoal:Existing loan takeover'), test_graph.attribute_relation('RiskClass'), test_graph.uri('RiskClass:'+seniority)))
    for res_index, resource in enumerate(['User_1', 'User_2', 'User_3']):
        test_result = validator.test_assignment(task_to_be_allocated, test_graph.uri('resource:'+resource))
        score, verdict = validator.calculate_score(test_result)
        if res_index >= sen_index:
            assert score > 0 
        else:
            assert score < 0 and verdict != '' 
        print((score, verdict))